# Stage 02: Real GEC pairs + N-best + error-signal report  `[CPU]`
Paper §3.1 — run Gipformer (or mock for smoke) over ViMedCSS audio to build
`raw_asr -> gold_text` pairs. `--n-best` adds the perturbation hypotheses (paper
§4.3). The **error-signal report** warns if the ASR is too accurate on train to
teach the corrector (paper §3.2).

In [ ]:
# --- CarePath stage bootstrap (short by design) ---
import importlib.util, os, subprocess, sys
from pathlib import Path

def _find(start):
    for d in [start, *start.parents]:
        if (d / 'pyproject.toml').exists() and (d / 'apps' / 'api' / 'carepath').exists():
            return d
    return None

REPO = _find(Path.cwd().resolve())
if REPO is None and importlib.util.find_spec('google.colab'):
    url = os.environ.get('CAREPATH_REPO_URL', 'https://github.com/truong-tt/carepath.git')
    tok = os.environ.get('CAREPATH_GITHUB_TOKEN') or os.environ.get('GITHUB_TOKEN')
    if tok and url.startswith('https://github.com/'):
        url = url.replace('https://', f'https://x-access-token:{tok}@')
    subprocess.run(['git', 'clone', url, '/content/carepath'], check=True)
    REPO = Path('/content/carepath')
assert REPO, 'Open this notebook from inside the CarePath repo.'
os.chdir(REPO); sys.path.insert(0, str(REPO / 'apps' / 'api'))

PROFILE = 'smoke'   # <<< set to 'full' for the real ViMedCSS run
from carepath.gec.notebook import init_stage
CTX = init_stage(PROFILE); P = CTX.paths; PROF = CTX.profile


In [ ]:
# Install the GEC training stack (idempotent; needed once per Colab runtime).
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[training]'])


In [ ]:
CTX.run_step(['scripts/gec/make_pairs.py', '--dataset', CTX.dataset, '--output', str(P.real_pairs),
              '--asr-provider', PROF.asr_provider, '--datastore', str(P.datastore),
              '--retrieval-backend', PROF.retrieval_backend,
              '--limit-per-split', str(PROF.limit_per_split or 0),
              '--n-best', str(PROF.n_best), '--resume'])
from carepath.gec.data import read_jsonl
from carepath.gec.evaluate import train_error_signal
print(train_error_signal(read_jsonl(P.real_pairs)))
CTX.save([str(P.datastore), str(P.real_pairs)])
